In [1]:
import pandas as pd

df = pd.read_csv("cafe_menu_processed.csv")

In [ ]:
from langchain_core.documents import Document

# 키워드 사전
flavor_keywords = [
    # 과일
    "딸기", "복숭아", "레몬", "자몽", "망고", "블루베리", "체리", "포도",
    "수박", "메론", "키위", "파인애플", "라즈베리", "패션후르츠", "유자",
    "청포도", "사과", "배", "귤", "오렌지", "라임", "코코넛", "바나나",
    "산딸기", "크랜베리", "무화과", "석류",

    # 맛/재료
    "초코", "초콜릿", "바닐라", "카라멜", "헤이즐넛", "시나몬", "민트",
    "꿀", "흑당", "흑임자", "말차", "녹차", "곡물", "쌀", "오레오",
    "크림", "치즈", "버터", "소금", "캐러멜", "아몬드", "피스타치오",
    "땅콩", "참깨", "콩", "팥", "쑥", "생강", "계피", "커피",
    "에스프레소", "연유", "요거트", "우유", "두유", "오트",

    # 시럽/토핑
    "펄", "타피오카", "젤리", "푸딩", "버블", "휘핑", "샷",
    "시럽", "잼", "소스",
]

base_keywords = [
    # 커피 베이스
    "아메리카노", "에스프레소", "라떼", "카푸치노", "마끼아또", "모카",
    "콜드브루", "드립", "플랫화이트", "리스트레토", "룽고",

    # 논커피 베이스
    "밀크티", "티", "허브티", "과일티", "블렌디드", "스무디", "에이드",
    "주스", "쉐이크", "프라푸치노", "요거트", "소다", "탄산",
    "코코아", "초코", "레모네이드", "하이볼",

    # 우유 종류
    "두유", "오트밀크", "아몬드밀크", "저지방",
]

def extract_keywords(name):
    found_flavors = [k for k in flavor_keywords if k in name]
    found_bases = [k for k in base_keywords if k in name]
    return found_flavors, found_bases

def row_to_sentence(row):
    coffee_str = "커피 음료" if row['is_coffee'] else "커피가 들어가지 않는 음료"
    temp_map = {
        "ICE": "차갑게",
        "HOT": "따뜻하게",
        "BOTH": "차갑게 또는 따뜻하게"
    }
    temp_str = temp_map.get(row['temp'], row['temp'])
    
    flavors, bases = extract_keywords(row['name'])
    
    # 키워드 문장 추가
    keyword_str = ""
    if flavors:
        keyword_str += f"이 음료의 맛/재료는 {', '.join(flavors)}입니다. "
    if bases:
        keyword_str += f"베이스 음료는 {', '.join(bases)}입니다. "
    if flavors and bases:
        keyword_str += f"한 줄 요약: {' + '.join(flavors + bases)}. "

    return (
        f"{row['brand']}에서 판매하는 {row['name']}은(는) "
        f"{row['category']} 카테고리에 속하는 {coffee_str}입니다. "
        f"이 음료는 {temp_str} 즐길 수 있습니다. "
        f"{keyword_str}"
    )

documents = []

for _, row in df.iterrows():
    flavors, bases = extract_keywords(row['name'])
    documents.append(
        Document(
            page_content=row_to_sentence(row),
            metadata={
                "brand": row["brand"],
                "name": row["name"],
                "temp": row["temp"],
                "category": row["category"],
                "is_coffee": row["is_coffee"],
                "flavors": flavors,    
                "bases": bases,
            }
        )
    )

In [5]:
documents[0]

Document(metadata={'brand': '공차', 'name': '딸기 쥬얼리 밀크티', 'temp': 'ICE', 'category': '밀크티', 'is_coffee': False, 'flavors': ['딸기'], 'bases': ['밀크티', '티']}, page_content='공차에서 판매하는 딸기 쥬얼리 밀크티은(는) 밀크티 카테고리에 속하는 커피가 들어가지 않는 음료입니다. 이 음료는 차갑게 즐길 수 있습니다. 이 음료의 맛/재료는 딸기입니다. 베이스 음료는 밀크티, 티입니다. 한 줄 요약: 딸기 + 밀크티 + 티. ')

In [ ]:
# 기존 데이터 삭제 후 재업로드
pc.Index("cafe-menu-index").delete(delete_all=True)
database = PineconeVectorStore.from_documents(documents, embedding, index_name=index_name)

In [ ]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone


embedding = OpenAIEmbeddings(model="text-embedding-3-small")  # 그대로

database = PineconeVectorStore.from_documents(
    documents=documents,
    embedding=embedding,
    index_name="cafe-menu-index"
)

In [ ]:
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'cafe-menu-index'
pc = Pinecone(api_key=pinecone_api_key)


database = PineconeVectorStore.from_documents(documents, embedding, index_name=index_name)